# Fault Tolerance Experiments: Empirical Resilience Verification

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 05_fault_tolerance_experiments.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Laboratory suite of empirical fault tolerance experiments: lineage reconstruction, task failure recovery, and checkpoint restoration.
- **Why does this topic exist?**: Proving resilience empirically prepares you for production chaos engineering.
- **What problem does it solve?**: Assuming fault tolerance works without verifying recovery behavior.
- **Where is it used in real-world Data Engineering?**: Disaster recovery testing for mission-critical pipelines.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Executor failure, task retries, checkpointing |
| **Required depth** | Deep working knowledge: can design and interpret resilience experiments |
| **Data Engineer** | Run controlled failure experiments and verify recovery behavior. |
| **Senior Data Engineer** | Build a chaos-testing practice for data pipelines. |
| **Spark specialist** | Reason about failure injection, observability, and recovery invariants. |
| **Next topic** | `12_advanced_topics/01_spark_on_yarn.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Fault tolerance is a claim until it is tested. A controlled experiment injects a specific failure, observes
how Spark reacts, and verifies that the output is still correct. The goal is not to break the cluster randomly; it
is to validate a recovery invariant, such as "a lost executor does not duplicate output" or "a checkpoint bounds
recomputation." 

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Failure injection** | Deliberate fault | Kill an executor, fail a task, remove a file |
| **Recovery invariant** | What must remain true | No data loss, no duplicates, bounded recomputation |
| **Observability** | Evidence of recovery | Stage retries, task attempts, checkpoint reads |
| **Blast radius** | Scope of the experiment | Isolate to a test job and dataset |
| **Rollback** | Restore normal state | Restart services, clean checkpoints |
| **Post-mortem** | Learn from the result | Document behavior and gaps |

### How it works, step by step

1. Define the invariant and the failure to inject.
2. Set up an isolated job with known input and output.
3. Capture baseline metrics and results.
4. Inject the failure at a controlled point.
5. Observe retries, recomputation, and final output.
6. Compare with the invariant and document the gap.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("FT_Experiments_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print(f"Loaded {orders.count()} orders for resilience experiments.")

---

## Experiment: Simulated Node Crash Recomputation

- **1. What are we testing?**: Unpersist an RDD and observe that subsequent actions re-evaluate from source files.
- **2. What do we expect?**: Recomputation succeeds seamlessly.

In [ ]:
rdd_test = orders.rdd.map(lambda r: (r[7], 1)).reduceByKey(lambda a, b: a+b).persist()
rdd_test.count() # materialize
rdd_test.unpersist() # simulate block loss
print("Count after simulated loss:", rdd_test.count())

**3. What happened?**: Spark seamlessly re-read source files and re-executed the pipeline.  
**4. Why did it happen?**: Lineage remained intact, allowing automatic recomputation.  
**5. Practical Engineering Lesson**: Spark handles in-memory block loss transparently through lineage graphs.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Chaos testing a nightly ETL

- **Industry**: SaaS
- **Scenario**: A platform team wants confidence that executor loss does not corrupt the nightly load.
- **Problem**: Untested recovery logic may duplicate or lose data under real failures.
- **Why this concept is relevant**: A controlled executor-kill experiment validates the invariant before an incident occurs.
- **Implementation idea**: Run the job on a test dataset, kill an executor mid-shuffle, and compare output keys with the baseline.
- **Expected result**: A documented recovery behavior and a fixed gap if found.
- **Alternative**: Rely on production incidents to discover gaps.
- **Trade-offs**: Production incidents are realistic but costly; controlled experiments are safer.

### Use Case 2: Validating a checkpoint strategy

- **Industry**: Financial services
- **Scenario**: A long risk pipeline adds checkpointing to reduce recovery time.
- **Problem**: The team assumes the checkpoint will be used, but never tests a failure after it.
- **Why this concept is relevant**: A failure experiment proves that recovery starts from the checkpoint and that the result is unchanged.
- **Implementation idea**: Kill the job after the checkpoint, restart, and measure recovery time and output equality.
- **Expected result**: A verified reduction in recovery time.
- **Alternative**: Measure only normal runtime.
- **Trade-offs**: Normal runtime does not reveal recovery behavior; the experiment is necessary.

---

## Business Value

- Confidence in recovery behavior before an incident.
- Evidence for SLA and RTO planning.
- Faster root-cause analysis during real failures.
- A culture of testing operational assumptions.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Test one failure at a time in an isolated environment.
- Verify output correctness, not just job completion.
- Record recovery time and recomputation cost.
- Clean up checkpoints and temporary state after the experiment.
- Document the observed behavior and any gaps.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Kill an executor and verify output

**Steps**

1. Run a job with a known output key set.
2. Kill one executor during the shuffle.
3. Let the job finish and compare the output keys and counts with the baseline.

**What to observe**: Spark retries tasks and may recompute a stage; the final output should match the baseline.

**Healthy result**: No missing or duplicate keys.

**Problematic result**: The output differs, indicating non-determinism or a non-idempotent sink.

**Engineering takeaway**: Recovery must be verified at the data level.

---

## Trade-offs and Alternatives

### Advantages

- Validates real behavior
- Finds gaps before production
- Builds operational confidence

### Limitations

- Requires isolation
- Can be time-consuming
- Not all failures are easy to inject

### When to use it

- Chaos engineering
- Disaster recovery drills
- Checkpoint validation

### When not to use it

- Random production failures
- Testing without an invariant
- Ignoring output correctness

### Alternatives inside Spark

- Executor kill
- Task failure
- Checkpoint recovery

### Alternatives outside Spark

- Chaos platforms
- Game days
- Backup/restore drills

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Verify Unpersist
**Problem Statement**: Check is_cached is False after unpersist.  
**Input / Context**: rdd_test

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Check Task Failures in UI
**Problem Statement**: Inspect failed tasks count in UI Executors tab.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Chaos Engineering Scenario
**Problem Statement**: Explain what happens if Master container dies.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The experiment that proved nothing

**Scenario**: An executor is killed after the job has already finished, and the team concludes recovery works.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Inject the failure while the job is actively processing.
2. Verify that the job actually used the executor.
3. Check the timing and the stage that was running.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Verify Unpersist):
```python
print('Is cached:', rdd_test.is_cached)
```

#### Solution 2 (Check Task Failures in UI):
```python
print('Inspected localhost:4040/executors')
```

#### Solution 3 (Chaos Engineering Scenario):
```python
print('Running applications continue; new submissions block until Master recovers.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Resilience Validated**: Lineage recomputes lost blocks; task retries absorb glitches.

### Essential Technical Terms
- `Chaos Engineering`: Testing systems by intentionally introducing failures.

### Common Engineering Mistakes
- **Assuming data loss on executor crash**: Lineage guarantees deterministic recovery.

### Interview Questions on This Topic
- **Q: Can Spark recover if the Driver dies in client mode?**
  - *A*: No, in client mode the driver hosts the DAGScheduler and SparkContext; if it dies, the application terminates.


### More Interview Questions

- **Q: What makes a good fault-tolerance experiment?**
  - *A*: A defined recovery invariant, an isolated workload, a controlled failure injection, and a correctness check of the final output.
- **Q: Why compare output keys, not just job status?**
  - *A*: A job can complete successfully while losing or duplicating data; key-level comparison verifies the actual invariant.

### What to Study Next
Module 11 complete! Proceed to Module 12: Advanced Topics.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')